# Agent Coding Task: Fix a Bug in a Sandbox

Drop a small Python repo into the sandbox, hand the agent shell + file
tools, and ask it to fix a failing test. Classic "AI software engineer"
loop - the agent reads `task.md`, edits the source, re-runs pytest, and
reports back.

## Prerequisites
- Azure CLI [signed in](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)
- `azure-containerapps-sandbox` SDK (from GitHub Releases - see [lab README](./README.md#prerequisites)):
  ```bash
  gh release download v0.1.0b1 \
      --repo Azure-Samples/azure-container-apps-sandboxes \
      --pattern "azure_containerapps_sandbox-*.whl" \
      --output azure_containerapps_sandbox.whl
  pip install azure_containerapps_sandbox.whl
  pip install openai-agents
  ```
- An OpenAI provider, configured **once** for the whole lab in `provider_config.py`
  (see the [lab README](./README.md#prerequisites)). Azure OpenAI is preferred and
  uses Entra ID auth by default.

Click **Run All** or go **Step by Step**.

### 0. Initialize variables

In [ ]:
import os, json, shutil, subprocess
from azure.containerapps.sandbox import SandboxClient, SandboxGroupClient

lab_name = os.path.basename(os.path.dirname(os.path.abspath(
    globals().get('__vsc_ipynb_file__', os.path.join(os.getcwd(), '__file__')))))

# Resolve full path to the az CLI (on Windows it's az.cmd, which subprocess.run
# can't find by bare name without shell=True).
az = shutil.which('az')
if not az:
    raise RuntimeError(
        'Azure CLI not found on PATH. Install it from '
        'https://learn.microsoft.com/cli/azure/install-azure-cli and re-run.'
    )

account = json.loads(subprocess.run(
    [az, 'account', 'show', '-o', 'json'],
    capture_output=True, text=True, check=True).stdout)

subscription_id = account['id']
resource_group_name = f'lab-{lab_name}'
resource_group_location = 'westus2'
sandbox_group_name = f'{lab_name}-sg'
repo_dir = '/home/user/repo'

print(f'Lab:            {lab_name}')
print(f'Resource Group: {resource_group_name}')
print(f'Sandbox Group:  {sandbox_group_name}')
print(f'Repo (in sbx):  {repo_dir}')

client = SandboxClient(subscription_id=subscription_id, resource_group=resource_group_name)
mgmt = SandboxGroupClient(subscription_id=subscription_id, resource_group=resource_group_name)

### 0a. Configure the OpenAI provider

Provider config is shared across the three notebooks in this lab. Copy [`provider_config.py.example`](./provider_config.py.example) to `provider_config.py` (gitignored) and fill in either the Azure OpenAI or OpenAI section -- all three notebooks pick it up automatically. If you prefer environment variables, leave it as-is and set `AZURE_OPENAI_ENDPOINT`/`AZURE_OPENAI_DEPLOYMENT` (Entra ID auth by default) or `OPENAI_API_KEY` in your shell before launching VS Code.

In [ ]:
import sys, os
_lab_dir = os.path.dirname(os.path.abspath(
    globals().get('__vsc_ipynb_file__', os.path.join(os.getcwd(), '__file__'))))
if _lab_dir not in sys.path:
    sys.path.insert(0, _lab_dir)

from provider import get_model
model = get_model()


### 1. Create the sandbox

In [ ]:
!az group create --name {resource_group_name} --location {resource_group_location} -o none

group = mgmt.create_group(sandbox_group_name, location=resource_group_location)
print(f'Group: {group.name}')

# Ensure the signed-in user has data-plane access on the sandbox group.
# `mgmt.create_group` only needs ARM (control plane) rights, but
# `client.create_sandbox` calls the data plane (management.azuredevcompute.io)
# which requires a separate role on the sandbox group resource itself.
sg_id = group.id
data_role = 'Dev Compute SandboxGroup Data Owner'

signed_in_oid = subprocess.run(
    [az, 'ad', 'signed-in-user', 'show', '--query', 'id', '-o', 'tsv'],
    capture_output=True, text=True, check=True).stdout.strip()

existing = json.loads(subprocess.run(
    [az, 'role', 'assignment', 'list',
     '--assignee', signed_in_oid,
     '--scope', sg_id,
     '--role', data_role,
     '-o', 'json'],
    capture_output=True, text=True, check=True).stdout)

if not existing:
    print(f'Assigning "{data_role}" to {signed_in_oid} on sandbox group...')
    subprocess.run(
        [az, 'role', 'assignment', 'create',
         '--assignee-object-id', signed_in_oid,
         '--assignee-principal-type', 'User',
         '--role', data_role,
         '--scope', sg_id,
         '-o', 'none'],
        check=True)
    # Role assignments take a few seconds to propagate to the data plane.
    import time
    for i in range(30):
        time.sleep(5)
        try:
            sbx = client.create_sandbox(sandbox_group_name, disk='python-3.13')
            break
        except Exception as e:
            if '403' not in str(e) or i == 29:
                raise
            print(f'  waiting for role propagation... ({(i+1)*5}s)')
    else:
        raise RuntimeError('Role assignment did not propagate within 150s')
else:
    print(f'Role "{data_role}" already assigned.')
    sbx = client.create_sandbox(sandbox_group_name, disk='python-3.13')

sandbox_id = sbx.id
print(f'Sandbox: {sandbox_id}')

### 2. Install pytest in the sandbox

We use the public `python-3.13` disk image (set in the previous cell), so Python is preinstalled. We just need to add `pytest`.

In [ ]:
print('Installing pytest in the sandbox...')
r = client.exec(sandbox_id, sandbox_group_name, 'pip install --quiet pytest')
if r.exit_code != 0:
    print('STDOUT:', r.stdout)
    print('STDERR:', r.stderr)
    raise RuntimeError(f'pytest install failed with exit code {r.exit_code}')
print('pytest installed.')


### 3. Seed the workspace with a buggy repo

Drop three files into the sandbox: `task.md` describes the bug,
`src/hello.py` has the bug, and `tests/test_hello.py` catches it.

In [ ]:
task_md = '''
# Greeting bug

The function `greet(name)` in `src/hello.py` is supposed to return
`"Hello, <name>!"`, but it currently returns `"Hi, <name>!"`. The pytest
suite under `tests/` expects the `Hello, <name>!` form and is failing.

## Goal

1. Read `src/hello.py` and `tests/test_hello.py`.
2. Fix the implementation in `src/hello.py` so it returns `Hello, <name>!`.
3. Verify the fix by running the targeted test:

   ```bash
   python -m pytest tests/test_hello.py -q
   ```

4. Summarize what you changed and the exact command you used to verify.
'''

hello_py = '''
"""Tiny module with a deliberately wrong greeting."""

from __future__ import annotations


def greet(name: str) -> str:
    """Return a greeting for ``name``.

    Should return ``"Hello, <name>!"`` but currently returns the wrong
    salutation. The test suite under ``tests/`` documents the correct
    behavior.
    """

    return f"Hi, {name}!"
'''

test_hello_py = '''
"""Tests for ``src/hello.py``.

The agent should make this file pass without modifying it.
"""

from __future__ import annotations

import sys
from pathlib import Path

# Make ``src`` importable when pytest is invoked from the repo root.
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from hello import greet  # noqa: E402


def test_greet_world() -> None:
    assert greet("world") == "Hello, world!"


def test_greet_handles_empty_string() -> None:
    assert greet("") == "Hello, !"
'''

client.exec(sandbox_id, sandbox_group_name, f'mkdir -p {repo_dir}/src {repo_dir}/tests')
client.write_file(sandbox_id, sandbox_group_name, f'{repo_dir}/task.md', task_md)
client.write_file(sandbox_id, sandbox_group_name, f'{repo_dir}/src/hello.py', hello_py)
client.write_file(sandbox_id, sandbox_group_name, f'{repo_dir}/tests/test_hello.py', test_hello_py)
print(f'Seeded buggy repo at {repo_dir}')

### 4. Define the agent's tools

Wrap the SDK as `@function_tool`s. The agent picks the tool, we forward
the call. We also pin every shell call to the repo directory so the agent
can use relative paths.

In [ ]:
from agents import Agent, Runner, function_tool

@function_tool
def shell(command: str) -> str:
    """Run a shell command inside the sandbox repo directory.

    The current working directory is always the repo root, so prefer
    relative paths (e.g. `cat src/hello.py`, `python -m pytest -q`).
    Returns the exit code, stdout, and stderr concatenated.
    """
    full = f'cd {repo_dir} && {command}'
    r = client.exec(sandbox_id, sandbox_group_name, full)
    return (
        f'exit_code={r.exit_code}\n'
        f'stdout:\n{r.stdout}\n'
        f'stderr:\n{r.stderr}'
    )

@function_tool
def read_file(path: str) -> str:
    """Read a UTF-8 text file. Path is relative to the repo root."""
    full = path if path.startswith('/') else f'{repo_dir}/{path}'
    return client.read_file(sandbox_id, sandbox_group_name, full).decode('utf-8', errors='replace')

@function_tool
def write_file(path: str, content: str) -> str:
    """Write text content to a file. Path is relative to the repo root."""
    full = path if path.startswith('/') else f'{repo_dir}/{path}'
    client.write_file(sandbox_id, sandbox_group_name, full, content)
    return f'wrote {len(content)} bytes to {full}'

@function_tool
def list_dir(path: str = '.') -> str:
    """List files in a directory. Path is relative to the repo root."""
    full = path if path.startswith('/') else f'{repo_dir}/{path}'
    r = client.exec(sandbox_id, sandbox_group_name, f'ls -la {full}')
    return r.stdout

agent = Agent(
    name='Sandbox Engineer',
    model=model,
    instructions=(
        'You are a careful software engineer with full access to a Linux '
        'sandbox via the `shell`, `read_file`, `write_file`, and `list_dir` '
        'tools. The current working directory is the repo root. Read '
        '`task.md` before editing. Stay grounded in the repo, preserve '
        'existing behavior, and mention the exact verification command '
        'you ran.'
    ),
    tools=[shell, read_file, write_file, list_dir],
)
print('Agent ready with', len(agent.tools), 'tools.')

### 5. Run the agent

In [ ]:
question = (
    'Open repo/task.md, fix the issue it describes, run the targeted test, '
    'and summarize what you changed and the exact verification command you ran.'
)

# Bumped max_turns so the agent has room to read, edit, run pytest, and reflect.
result = await Runner.run(agent, question, max_turns=20)
print(result.final_output)

### 6. Verify the fix independently

Run pytest one more time outside the agent loop to be sure the agent
really fixed the bug.

In [ ]:
verify = client.exec(sandbox_id, sandbox_group_name,
                     f'cd {repo_dir} && python -m pytest tests/test_hello.py -q')
print(f'exit_code={verify.exit_code}')
print(verify.stdout)
print(verify.stderr)
assert verify.exit_code == 0, 'pytest still failing — agent did not fix the bug'
print('OK: tests pass.')

### 7. Clean up

In [ ]:
client.delete_sandbox(sandbox_id, sandbox_group_name)
print('Deleted sandbox')

mgmt.delete_group(sandbox_group_name)
print('Deleted group')

!az group delete --name {resource_group_name} --yes --no-wait
print('Deleting resource group (async)')